###Initialization of libraries and path

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *

CATALOG = "quickcart"

SOURCE_VOLUME = f"/Volumes/{CATALOG}/default/source_data"

BRONZE_SCHEMA = "bronze"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.bronze;

###Building the function for ingestion

In [0]:
def ingest_to_bronze(source_table, source_format = 'parquet'):
    source_path = f"{SOURCE_VOLUME}/{source_table}"
    schema_path = f"{SOURCE_VOLUME}/_schemas/{source_table}"
    checkpoint_path = f"{SOURCE_VOLUME}/_checkpoints/{source_table}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{source_table}"

    print("=" * 70)
    print(f"Starting Bronze ingestion: {source_table}")
    print("=" * 70)
    
    print(f"Source Path     : {source_path}")
    print(f"Schema Location : {schema_path}")
    print(f"Checkpoint      : {checkpoint_path}")
    print(f"Target Table    : {target_table}")

    try:
            df = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format",source_format)\
                    .option("cloudFiles.schemaLocation",schema_path)\
                        .load(source_path)

            df = (df.withColumn("_ingestion_timestamp",F.current_timestamp())\
                .withColumn("_source_file",F.col("_metadata.file_path"))          
                )
            
            query = df.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation", checkpoint_path)\
                    .trigger(availableNow = True)\
                        .toTable(target_table)
            
            query.awaitTermination()

            print(
                f"Bronze ingestion completed: {target_table}"
            )

            return {
            "source_table": source_table,
            "target_table": target_table,
            "status": "SUCCESS",
            "error_message": None
            }

    except Exception as e:
            print(f"FAILED: Bronze ingestion failed "
            f"for {source_table}")
            print("Error:")
            print(str(e))
            return {
            "source_table": source_table,
            "target_table": target_table,
            "status": "FAILED",
            "error_message": str(e)
            }

In [0]:
ingest_to_bronze("deliveries")

In [0]:
%sql
SELECT * FROM quickcart.bronze.payments
LIMIT 10;